# 🏭 YOUAI AIStudio - Colab 多模态算力车间 (Multimodal GPU Worker v2.0)

本 Notebook 支持两种运行模式：
1. **Antigravity 云端全自动无头调度 (`colab-operator` CLI)**：由 [`scripts/colab_multimodal_worker.py`](../scripts/colab_multimodal_worker.py) 通过 `colab run` / `colab exec` 直接在云端 GPU 执行。
2. **Google Colab 网页端交互运行**：按顺序执行下方代码单元格，自动读取并回写 Google Drive 共享任务目录。

### 核心修复与工业级标准 (v2.0)
- **ASR 逐字时间戳 (`faster-whisper` large-v3)**：输出毫秒级 `word_timestamps.json`（含 `words` 与 `segments`）。
- **TTS 语音合成 (`F5-TTS` / `edge-tts` 双擎自适应)**：强制重采样为 `24kHz / 16-bit / mono PCM WAV`，并在首尾施加 `30ms` 淡入淡出防爆音滤波 (`afade`)。
- **T2I 视觉分镜生成 (`SDXL-Turbo`)**：修复 `guidance_scale=0.0`（杜绝过曝伪影），并以 `LANCZOS` 超采样输出 `1920x1080` 或 `1080x1920` 全高清分镜图。

In [ ]:
# @title 1. 环境初始化与依赖安装 (Install Dependencies)
!pip install -q faster-whisper f5-tts edge-tts diffusers transformers accelerate torch torchaudio pillow soundfile
print('✅ 依赖安装完成！')

In [ ]:
# @title 2. 挂载 Google Drive 与配置任务目录 (Mount Drive & Configure Job Dir)
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f'⚠️ 非交互式 Colab 环境或已挂载: {e}')

PROJECT_NAME = '20261107_jensen_power'  # @param {type:"string"}
WORK_DIR = Path(f'/content/drive/MyDrive/AIStudio_Jobs/{PROJECT_NAME}')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f'📂 当前任务工作目录: {WORK_DIR}')

In [ ]:
# @title 3. 执行多模态生产流水线 (ASR + TTS + SDXL-Turbo 1080p)
import json, subprocess
import torch
from PIL import Image

# --- 3.1 Faster-Whisper 逐字级 ASR ---
raw_audio_path = WORK_DIR / 'raw_audio.wav'
if raw_audio_path.exists():
    from faster_whisper import WhisperModel
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    compute_type = 'float16' if device == 'cuda' else 'int8'
    whisper_model = WhisperModel('large-v3', device=device, compute_type=compute_type)
    segments, info = whisper_model.transcribe(str(raw_audio_path), word_timestamps=True)
    words_out, segments_out = [], []
    for seg in segments:
        seg_words = []
        for w in (seg.words or []):
            item = {'word': w.word.strip(), 'start': round(w.start, 3), 'end': round(w.end, 3), 'probability': round(w.probability, 4)}
            words_out.append(item)
            seg_words.append(item)
        segments_out.append({'start': round(seg.start, 3), 'end': round(seg.end, 3), 'text': seg.text.strip(), 'words': seg_words})
    with open(WORK_DIR / 'word_timestamps.json', 'w', encoding='utf-8') as f:
        json.dump({'language': info.language, 'duration': round(info.duration, 3), 'words': words_out, 'segments': segments_out}, f, ensure_ascii=False, indent=2)
    print(f'✅ ASR 完成: 共提取 {len(words_out)} 个逐字时间戳')

# --- 3.2 SDXL-Turbo 无伪影 1080p 视觉生成 (guidance_scale=0.0) ---
prompts_file = WORK_DIR / 'visual_prompts.json'
if prompts_file.exists():
    from diffusers import AutoPipelineForText2Image
    with open(prompts_file, 'r', encoding='utf-8') as f:
        prompts = json.load(f)
    out_img_dir = WORK_DIR / 'generated_plates'
    out_img_dir.mkdir(exist_ok=True)
    pipe = AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16').to('cuda')
    for idx, item in enumerate(prompts):
        shot_id = item.get('id', f'shot_{idx:03d}')
        prompt_str = f"{item.get('prompt', '')}, cinematic lighting, 8k resolution, photorealistic, highly detailed"
        aspect = item.get('aspect', '16:9')
        gen_w, gen_h = (576, 1024) if aspect == '9:16' else (1024, 576)
        target_w, target_h = (1080, 1920) if aspect == '9:16' else (1920, 1080)
        img = pipe(prompt=prompt_str, num_inference_steps=4, guidance_scale=0.0, width=gen_w, height=gen_h).images[0]
        img = img.resize((target_w, target_h), Image.Resampling.LANCZOS)
        img.save(out_img_dir / f'{shot_id}.png')
        print(f'✅ 生成 1080p 分镜图: {shot_id}.png ({target_w}x{target_h})')

with open(WORK_DIR / '_COLAB_DONE', 'w') as f:
    f.write('SUCCESS')
print('🎉 多模态算力车间任务全部完成！')